In [6]:
import os
import numpy as np
from PIL import Image
dataset_path=r"C:\Users\pshar\Downloads\RIM-ONE_DL_images\RIM-ONE_DL_images\partitioned_by_hospital"
def load_images(folder,label):
    images=[]
    labels=[]
    for file in os.listdir(folder):
        path=os.path.join(folder,file)
        try:
            image=Image.open(path)
            image=image.convert("L")
            image=image.resize((32,32))
            image=np.array(image)
            image=image.flatten()
            image=image/255.0
            images.append(image)
            labels.append(label)
        except:
            pass
    return images,labels
normal_train=os.path.join(dataset_path,"train_set","normal")
glaucoma_train=os.path.join(dataset_path,"train_set","glaucoma")
normal_test=os.path.join(dataset_path,"test_set","normal")
glaucoma_test=os.path.join(dataset_path,"test_set","glaucoma")

normal1,label1=load_images(normal_train,0)
glaucoma1,label2=load_images(glaucoma_train,1)
normal2,label3=load_images(normal_test,0)
glaucoma2,label4=load_images(glaucoma_test,1)
X=normal1+glaucoma1+normal2+glaucoma2
y=label1+label2+label3+label4
X=np.array(X,dtype=float)
y=np.array(y)
print("Total images:",len(X))
print("Normal images:",np.sum(y==0))
print("Glaucoma images:",np.sum(y==1))
def impute_data(X,method="mean"):
    X=X.copy()
    for column in range(X.shape[1]):
        missing=np.isnan(X[:,column])
        if np.any(missing):
            if method=="mean":
                value=np.nanmean(X[:,column])
            elif method=="median":
                value=np.nanmedian(X[:,column])
            X[missing,column]=value
    return X
X=impute_data(X,"mean")
def encode_labels(y):
    result=[]
    for value in y:
        if value==0:
            result.append(0)
        elif value==1:
            result.append(1)
    return np.array(result)
y=encode_labels(y)
def calculate_distance(a,b,method):
    if method=="euclidean":
        return np.sqrt(np.sum((a-b)**2))
    elif method=="manhattan":
        return np.sum(np.abs(a-b))
    elif method=="minkowski":
        p=3
        return (np.sum(np.abs(a-b)**p))**(1/p)
def bubble_sort(data):
    data=data.copy()
    for i in range(len(data)):
        for j in range(0,len(data)-i-1):
            if data[j][0]>data[j+1][0]:
                data[j],data[j+1]=data[j+1],data[j]
    return data
def selection_sort(data):
    data=data.copy()
    for i in range(len(data)):
        minimum=i
        for j in range(i+1,len(data)):
            if data[j][0]<data[minimum][0]:
                minimum=j
        data[i],data[minimum]=data[minimum],data[i]
    return data
def insertion_sort(data):
    data=data.copy()
    for i in range(1,len(data)):
        current=data[i]
        j=i-1
        while j>=0 and data[j][0]>current[0]:
            data[j+1]=data[j]
            j=j-1
        data[j+1]=current
    return data
sorting_method="bubble"
def sort_data(data,method):
    if method=="bubble":
        return bubble_sort(data)
    elif method=="selection":
        return selection_sort(data)
    elif method=="insertion":
        return insertion_sort(data)
def find_neighbors(X_train,y_train,test_point,k,distance_method,sorting_method):
    distances=[]
    for i in range(len(X_train)):
        distance=calculate_distance(X_train[i],test_point,distance_method)
        distances.append((distance,y_train[i]))
    distances=sort_data(distances,sorting_method)
    return distances[:k]
def majority_vote(neighbors):
    votes={}
    for distance,label in neighbors:
        if label not in votes:
            votes[label]=0
        votes[label]+=1
    highest=max(votes.values())
    winners=[]
    for label in votes:
        if votes[label]==highest:
            winners.append(label)
    if len(winners)>1:
        distance_sum={}
        for label in winners:
            distance_sum[label]=0
            for distance,neighbor_label in neighbors:
                if neighbor_label==label:
                    distance_sum[label]+=distance
        return min(winners,key=lambda x:distance_sum[x])
    return winners[0]
def knn_predict(X_train,y_train,X_test,k,distance_method,sorting_method):
    predictions=[]
    for test_point in X_test:
        neighbors=find_neighbors(X_train,y_train,test_point,k,distance_method,sorting_method)
        prediction=majority_vote(neighbors)
        predictions.append(prediction)
    return np.array(predictions)

Folder not found: C:\Users\pshar\Downloads\RIM-ONE_DL_images\RIM-ONE_DL_images\partitioned_by_hospital\train_set\normal
Folder not found: C:\Users\pshar\Downloads\RIM-ONE_DL_images\RIM-ONE_DL_images\partitioned_by_hospital\train_set\glaucoma
Total images: 174
Normal images: 118
Glaucoma images: 56


In [8]:
def weighted_vote(neighbors):
    weights={}
    for distance,label in neighbors:
        weight=1/(distance+0.000001)
        if label not in weights:
            weights[label]=0
        weights[label]+=weight
    highest=max(weights.values())
    winners=[]
    for label in weights:
        if weights[label]==highest:
            winners.append(label)
    if len(winners)>1:
        return min(winners)
    return winners[0]
def weighted_knn_predict(X_train,y_train,X_test,k,distance_method,sorting_method):
    predictions=[]
    for test_point in X_test:
        neighbors=find_neighbors(X_train,y_train,test_point,k,distance_method,sorting_method)
        prediction=weighted_vote(neighbors)
        predictions.append(prediction)
    return np.array(predictions)

In [11]:
import numpy as np
from sklearn.model_selection import train_test_split
X=np.array(X)
y=np.array(y)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3)
print("X_train:",X_train.shape)
print("X_test:",X_test.shape)
print("y_train:",y_train.shape)
print("y_test:",y_test.shape)

X_train: (121, 1024)
X_test: (53, 1024)
y_train: (121,)
y_test: (53,)


In [12]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
neigh=KNeighborsClassifier(n_neighbors=3)
neigh.fit(X_train,y_train)

,n_neighbors,3
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [13]:
accuracy=neigh.score(X_test,y_test)
print("Accuracy=",accuracy)
print("Accuracy=",accuracy*100,"%")

Accuracy= 0.6226415094339622
Accuracy= 62.264150943396224 %


In [14]:
prediction=neigh.predict(X_test)
print(prediction)

[0 0 1 1 0 0 1 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 1 0 1 1 0 1 1 0 0 1
 1 0 1 1 1 1 0 0 0 0 0 0 0 0 0 0]


In [15]:
class MyKNN:
    def __init__(self,k=3):
        self.k=k
        self.X_train=None
        self.y_train=None
    def Fit(self,X,y):
        self.X_train=X
        self.y_train=y
    def Predict(self,X):
        return knn_predict(self.X_train,self.y_train,X,self.k,"euclidean","bubble")
    def Score(self,X,y):
        predictions=self.Predict(X)
        correct=np.sum(predictions==y)
        return correct/len(y)
model=MyKNN(k=3)
model.Fit(X_train,y_train)
prediction=model.Predict(X_test)
accuracy=model.Score(X_test,y_test)
print("Predictions=",prediction)
print("Accuracy=",accuracy)
print("Accuracy=",accuracy*100,"%")

Predictions= [0 0 1 1 0 0 1 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 1 0 1 1 0 1 1 0 0 1
 1 0 1 1 1 1 0 0 0 0 0 0 0 0 0 0]
Accuracy= 0.6226415094339622
Accuracy= 62.264150943396224 %
